# 01 · Corpus preparation and pretraining a tiny character model

*Large Language Model Training*, notebook 1 of 4 · lessons 1 and 2: **Build a versioned and auditable text corpus** and **Plan base-model pretraining**

A language model can only learn what its corpus contains, so the corpus deserves the same care as the code. In this notebook you clean a small generated corpus into which we have deliberately put exact copies, near-copies and sentences leaked from an evaluation set. You then plan a pretraining run with the standard compute estimate and pretrain a tiny character-level transformer on the clean text. The model has about 29,000 parameters and the corpus about 36,000 characters, so the notebook teaches the *mechanics* of preparing data and pretraining. It says nothing about the language ability of real models, which are many thousands of times larger and read many millions of times more text.

## What you will build

- `content_hash(text)`: a fingerprint of normalised text, to find exact duplicates;
- `char_shingles(text, k)` and `jaccard(a, b)`: overlapping character windows and a set similarity, to find near-duplicates;
- `word_ngrams(text, n)`: runs of words that reveal evaluation sentences hiding in the training data;
- `training_budget(...)`: the $6ND$ compute estimate, tokens per parameter and wall-clock hours;
- a split by document, a packed token stream and a 300-step pretraining run, with validation loss reported for each source.

## Learning objectives

By the end of this notebook you will be able to:

1. explain why text is normalised before hashing, and remove exact duplicates while keeping a record of what replaced what;
2. measure near-duplication with character shingles and Jaccard similarity, and choose a threshold from measured catches and false alarms;
3. check a training corpus for contamination by an evaluation set, and say what an n-gram check cannot see;
4. estimate training compute with $C \approx 6ND$, and reason about tokens per parameter and repeated data;
5. split by document before cutting token windows, pack documents with an end-of-document token, and monitor validation loss for each source separately.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for the arrays and the model, and Matplotlib for one plot of the loss curves. From the standard library, `hashlib` computes SHA-256 fingerprints and `re` splits text into words. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the corpus is generated in the notebook from fixed seeds, so your numbers should match the ones quoted in the text up to small differences between computers.
- The model is trained with the tiny tensor engine from Base Model Training, pasted into a hidden recap cell, together with a small causal transformer and the Adam optimiser written for that engine.
- The whole notebook runs in a few seconds on a laptop; in the browser it takes longer, mostly in the pretraining cell.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import hashlib
import re
import time
from collections import Counter

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell defines the model this course trains, written with the tensor engine: a causal transformer of the kind you built in the Transformers course, kept as small as possible.

- `init_tiny_gpt(seed)` returns a dict of named weights: the token embeddings `tok` (vocabulary × 32) and position embeddings `pos` (32 × 32); two pre-norm blocks, `b0.*` and `b1.*`, each with attention projections `Wq`, `Wk`, `Wv`, `Wo` (32 × 32), two attention heads, and an MLP with `W1` (32 × 128) and `W2` (128 × 32); a final layer norm; and the output projection `Wout` (32 × vocabulary).
- `gpt_forward(params, ids)` maps token ids of shape `(batch, T)` to next-token logits of shape `(batch, T, vocabulary)`. A causal mask lets position $t$ see only positions $\le t$. Every weight matrix is applied through `project(x, params, name)`, which notebook 03 extends with adapters.
- `Adam(tensors, lr)` updates a list of tensors after `loss.backward()`, with bias correction and clipping of the overall gradient norm at 1.

The model needs the vocabulary, which you meet in section 5, before it can be built.

In [ ]:
# 📦 Recap: the tiny causal transformer and Adam, written with the tensor engine.

def project(x, params, name, adapters=None):
    """x @ W for the named weight matrix. (Notebook 03 attaches low-rank adapters here.)"""
    return x @ params[name]


# ── A tiny causal transformer, written with the tensor engine ───────────────
# ids (batch, T) -> embeddings (batch, T, D) -> logits (batch, T, len(VOCAB)).
# Each block is pre-norm: x + attention(layer_norm(x)), then x + mlp(layer_norm(x)).
CONTEXT, D_MODEL, N_HEADS, N_LAYERS, D_FF = 32, 32, 2, 2, 128


def init_tiny_gpt(seed=0):
    """Every weight of the model, as a dict name -> trainable Tensor."""
    init = np.random.default_rng(seed)
    D = D_MODEL
    params = {"tok": parameter(len(VOCAB), D, rng=init, scale=0.1),
              "pos": parameter(CONTEXT, D, rng=init, scale=0.1)}
    for b in range(N_LAYERS):
        params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        for name in ("Wq", "Wk", "Wv"):
            params[f"b{b}.{name}"] = parameter(D, D, rng=init, scale=D ** -0.5)
        params[f"b{b}.Wo"] = parameter(D, D, rng=init, scale=(2 * N_LAYERS * D) ** -0.5)
        params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        params[f"b{b}.W1"], params[f"b{b}.b1"] = parameter(D, D_FF, rng=init, scale=D ** -0.5), zeros_parameter(D_FF)
        params[f"b{b}.W2"] = parameter(D_FF, D, rng=init, scale=(2 * N_LAYERS * D_FF) ** -0.5)
        params[f"b{b}.b2"] = zeros_parameter(D)
    params["lnf_g"], params["lnf_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
    params["Wout"] = parameter(D, len(VOCAB), rng=init, scale=D ** -0.5)
    return params


def gpt_forward(params, ids, adapters=None):
    """Next-token logits (batch, T, len(VOCAB)) for integer token ids (batch, T), with T <= CONTEXT."""
    ids = np.asarray(ids)
    batch, T = ids.shape
    D = params["tok"].shape[1]
    hd = D // N_HEADS
    future = np.triu(np.full((T, T), -1e9), 1)  # position t may not look at positions after t
    x = params["tok"][ids] + params["pos"][np.arange(T)]
    for b in range(N_LAYERS):
        h = layer_norm(x, params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"])
        q, k, v = (project(h, params, f"b{b}.{name}", adapters).reshape(batch, T, N_HEADS, hd).transpose(0, 2, 1, 3)
                   for name in ("Wq", "Wk", "Wv"))
        attention = ((q @ k.transpose(0, 1, 3, 2)) * hd ** -0.5 + future).softmax(axis=-1)
        mixed = (attention @ v).transpose(0, 2, 1, 3).reshape(batch, T, D)
        x = x + project(mixed, params, f"b{b}.Wo", adapters)
        h = layer_norm(x, params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"])
        hidden = (project(h, params, f"b{b}.W1", adapters) + params[f"b{b}.b1"]).relu()
        x = x + project(hidden, params, f"b{b}.W2", adapters) + params[f"b{b}.b2"]
    return project(layer_norm(x, params["lnf_g"], params["lnf_b"]), params, "Wout", adapters)


class Adam:
    """Adam with bias correction and clipping of the global gradient norm, for a list of Tensors."""

    def __init__(self, tensors, lr=3e-3, betas=(0.9, 0.99), eps=1e-8, clip=1.0):
        self.tensors, self.lr, self.betas, self.eps, self.clip = list(tensors), lr, betas, eps, clip
        self.m = [np.zeros_like(t.data) for t in self.tensors]
        self.v = [np.zeros_like(t.data) for t in self.tensors]
        self.t = 0

    def step(self):
        self.t += 1
        b1, b2 = self.betas
        grads = [np.zeros_like(t.data) if t.grad is None else t.grad for t in self.tensors]
        norm = float(np.sqrt(sum((g ** 2).sum() for g in grads)))
        shrink = min(1.0, self.clip / (norm + 1e-12))
        for t, g, m, v in zip(self.tensors, grads, self.m, self.v):
            g = g * shrink
            m[...] = b1 * m + (1 - b1) * g
            v[...] = b2 * v + (1 - b2) * g * g
            t.data -= self.lr * (m / (1 - b1 ** self.t)) / (np.sqrt(v / (1 - b2 ** self.t)) + self.eps)
        return norm

## 1 · A small corpus with deliberate problems

Real pretraining corpora are scraped, licensed or donated text with every problem you can imagine. Ours is generated, so that you can see each problem clearly and *measure* whether your tools catch it. The toy world has three **sources**:

- `stories`: two to four sentences such as *the grey owl slept near the pond.*;
- `letters`: letter games on random three-letter strings, such as *kqo backwards is oqk.* or *say yad twice: yad yad.*;
- `lists`: shopping lists such as *list: two figs, five nuts.*

Before collecting any data we set aside an **evaluation set**: 40 reserved three-letter strings, and for each one the sentence *s backwards is (s reversed).* The letter games never use these strings or their reversals (the answers). Later notebooks test the instruction-tuned model on exactly these 40 strings, so none of them may appear in training.

`make_raw_corpus` then pollutes the clean corpus in the ways real corpora are polluted: 30 exact copies (some upper-cased or re-spaced), 20 near-copies (one word changed, or a short sentence added), four documents that quote an evaluation sentence word for word, and one that states an evaluation fact in different words. The returned `injected` dictionary records every one of them. With real data you would have no such record; here it lets you measure your detectors. The document ids tell the same story: `d…` generated, `x…` exact copies, `n…` near-copies, `q…` leaks.

In [ ]:
# ── The toy world: three sources of generated text (notebook 01) ────────────
LETTERS = "abcdefghijklmnopqrstuvwxyz"
ANIMALS = ["fox", "owl", "cat", "hen", "elk", "bee", "yak", "eel", "ram", "doe"]
COLOURS = ["red", "tan", "grey", "brown", "white", "black", "gold"]
VERBS = ["ran", "sat", "slept", "hid", "sang", "swam", "ate"]
PREPOSITIONS = ["to", "by", "near", "in"]
PLACES = ["river", "forest", "hill", "barn", "pond", "field", "lake"]
ITEMS = ["figs", "plums", "nuts", "pears", "eggs", "beans", "limes"]
NUMBERS = ["one", "two", "three", "four", "five", "six"]


def evaluation_strings(count=40, seed=123):
    """The reserved three-letter strings of the held-out evaluation set, sorted."""
    picker = np.random.default_rng(seed)
    chosen = set()
    while len(chosen) < count:
        chosen.add("".join(picker.choice(list(LETTERS), 3)))
    return sorted(chosen)


EVAL_STRINGS = evaluation_strings()
RESERVED = set(EVAL_STRINGS) | {s[::-1] for s in EVAL_STRINGS}  # no training text may contain these


def random_string(picker, banned=()):
    """A random three-letter string that is not in `banned`."""
    while True:
        s = "".join(picker.choice(list(LETTERS), 3))
        if s not in banned:
            return s


def make_document(source, picker, banned=()):
    """One generated document from 'stories', 'letters' or 'lists'."""
    if source == "stories":
        return " ".join(
            f"the {picker.choice(COLOURS)} {picker.choice(ANIMALS)} {picker.choice(VERBS)} "
            f"{picker.choice(PREPOSITIONS)} the {picker.choice(PLACES)}."
            for _ in range(picker.integers(2, 5))
        )
    if source == "letters":
        sentences = []
        for _ in range(picker.integers(2, 5)):
            s = random_string(picker, banned)
            sentences.append([f"{s} backwards is {s[::-1]}.", f"{s} starts with {s[0]}.",
                              f"{s} ends with {s[-1]}.", f"say {s} twice: {s} {s}."][picker.integers(4)])
        return " ".join(sentences)
    picks = picker.choice(ITEMS, picker.integers(2, 4), replace=False)
    return "list: " + ", ".join(f"{picker.choice(NUMBERS)} {item}" for item in picks) + "."


def make_clean_corpus(n_docs=500, seed=7):
    """Generated documents with ids, sources and text. Letter games never use a reserved string."""
    picker = np.random.default_rng(seed)
    docs = []
    for i in range(n_docs):
        source = str(picker.choice(["stories", "letters", "lists"], p=[0.45, 0.4, 0.15]))
        docs.append({"doc_id": f"d{i:04d}", "source": source,
                     "text": make_document(source, picker, RESERVED)})
    return docs

In [ ]:
def make_raw_corpus(n_docs=500, seed=7):
    """The clean corpus plus injected copies, near-copies and leaks. Returns (docs, injected)."""
    docs = make_clean_corpus(n_docs, seed)
    picker = np.random.default_rng(seed + 1)
    injected = {"exact": {}, "near": {}, "leaks": [], "paraphrase": None}
    for j in range(30):  # exact copies, some with different case or spacing
        original = docs[picker.integers(n_docs)]
        text = [original["text"], original["text"].upper(), "  " + original["text"].replace(" ", "   ") + "\n"][j % 3]
        docs.append({"doc_id": f"x{j:03d}", "source": original["source"], "text": text})
        injected["exact"][f"x{j:03d}"] = original["doc_id"]
    for j in range(20):  # near copies: one word changed, or a short sentence added
        original = docs[picker.integers(n_docs)]
        words = original["text"].split(" ")
        if j % 2 == 0:
            words[picker.integers(len(words))] = "big"
            text = " ".join(words)
        else:
            text = original["text"] + " the end."
        docs.append({"doc_id": f"n{j:03d}", "source": original["source"], "text": text})
        injected["near"][f"n{j:03d}"] = original["doc_id"]
    for j in range(4):  # leaks: a letter game that quotes an evaluation sentence word for word
        s = EVAL_STRINGS[7 * j]
        text = make_document("letters", picker, RESERVED) + f" {s} backwards is {s[::-1]}."
        docs.append({"doc_id": f"q{j:03d}", "source": "letters", "text": text})
        injected["leaks"].append(f"q{j:03d}")
    s = EVAL_STRINGS[30]  # a paraphrased leak: the same fact in different words
    docs.append({"doc_id": "q004", "source": "letters", "text": f"the reverse of {s} is {s[::-1]}."})
    injected["paraphrase"] = "q004"
    order = picker.permutation(len(docs))
    return [docs[i] for i in order], injected

In [ ]:
docs, injected = make_raw_corpus()
print(f"{len(docs)} documents, {sum(len(d['text']) for d in docs):,} characters")
print("documents per source:", dict(Counter(d["source"] for d in docs)))
print()
for d in docs[:6]:
    print(f"{d['doc_id']}  {d['source']:<8} {d['text'][:72]}")
print()
print("the first 8 of the 40 reserved evaluation strings:", EVAL_STRINGS[:8])

**What to notice.** Each record carries an id and a source as well as the text. A real manifest adds the acquisition date, the licence or permission to use the text, its language, and the version of every transform applied to it; below you add a fingerprint and a split. Keeping these fields from the start is what makes a corpus auditable later.

## 2 · Exact duplicates: normalise, then hash

Comparing every pair of documents character by character is slow and leaves nothing to record. Instead, compute a **fingerprint** of each document with a cryptographic hash function such as SHA-256, which maps any text to a 256-bit digest, written as 64 hexadecimal characters. Equal inputs give equal digests; a one-character change gives an unrelated digest; and the chance that two different documents share a digest is so small (below $2^{-128}$ per pair) that it can be ignored. Grouping documents by digest finds every exact duplicate in one pass.

"Exact" needs a definition first. *The red fox.* and *THE  RED FOX.* differ byte for byte but carry the same text, so we **normalise** before hashing: lower-case the text and collapse every run of whitespace into one space. Normalisation is a policy, not a fact. Record it, with a version number, next to the digests, because a different policy finds different duplicates.

The demo shows that the raw texts hash differently while their normalised forms are equal.

In [ ]:
def normalise(text):
    """Lower-case, and collapse every run of whitespace into one space (which also trims the ends)."""
    return " ".join(text.lower().split())


a, b = "The red fox sat.", "  the red   FOX sat.\n"
print(hashlib.sha256(a.encode("utf-8")).hexdigest()[:16], "  first 16 of 64 hex characters of the digest of a")
print(hashlib.sha256(b.encode("utf-8")).hexdigest()[:16], "  ... and of b")
print(repr(normalise(a)), repr(normalise(b)), normalise(a) == normalise(b))

### ✍️ Your turn: `content_hash`

Write `content_hash(text)`, which returns the SHA-256 digest of the **normalised** text as a string of 64 hexadecimal characters.

- Input: `text`, a string.
- Output: `hashlib.sha256(...).hexdigest()` of `normalise(text)` encoded as UTF-8 bytes.

Hints: `normalise` is defined above, and a string becomes bytes with `.encode("utf-8")`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below; open it only when you are stuck.

In [ ]:
def content_hash_reference(text):
    return hashlib.sha256(normalise(text).encode("utf-8")).hexdigest()

In [ ]:
def content_hash(text):
    """SHA-256 hex digest (64 characters) of normalise(text), encoded as UTF-8."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
content_hash = practice("content_hash", content_hash, content_hash_reference,
                        cases=[("The red fox sat.",), ("  the RED   fox sat.\n",), ("",)])

**Expected behaviour.** The first two cases return the same 64-character digest, because they normalise to the same text. The empty string has a digest too (`e3b0c442…`), so an empty document is a duplicate of every other empty document, which is what you want.

Now group the documents by digest. Within a group we keep the document with the **smallest id**, a fixed rule that keeps the original whenever one exists, and record every removal in `replaced_by`.

In [ ]:
groups = {}  # digest -> ids of the documents with that normalised text
for doc in docs:
    groups.setdefault(content_hash(doc["text"]), []).append(doc["doc_id"])

replaced_by = {}  # removed doc_id -> the doc_id kept in its place: the audit trail
for ids in groups.values():
    keep = min(ids)
    for other in ids:
        if other != keep:
            replaced_by[other] = keep

docs_unique = sorted((d for d in docs if d["doc_id"] not in replaced_by), key=lambda d: d["doc_id"])
print(f"{len(docs)} -> {len(docs_unique)} documents: {len(replaced_by)} exact duplicates removed")
print("injected copies caught:", sum(c in replaced_by for c in injected["exact"]), "of", len(injected["exact"]))
print("duplicates the generator produced by chance:", sorted(i for i in replaced_by if i.startswith("d")))
for removed in sorted(replaced_by)[-3:]:
    print(f"  {removed} replaced by {replaced_by[removed]}")

**What to notice.** "Keep the first one" after a shuffle keeps whichever copy happens to come first, often the upper-cased or re-spaced one. A fixed rule keeps the original, and `replaced_by` records every decision, so you can later answer the question "where did document `x007` go?". Templated sources also produce exact duplicates on their own: the shopping lists are short, so two of them can coincide by chance.

## 3 · Near-duplicates: shingles and Jaccard similarity

A copy with one word changed has a completely different digest, yet for training it is still the same document: repeated near-copies push the model to memorise them, and a near-copy on both sides of a split makes held-out text look easier than it is. To measure *how much* two texts overlap, cut each into **shingles**, the set of all its overlapping substrings of $k$ characters, and compare the two sets with the **Jaccard similarity**

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|},$$

which is 1 for identical sets and 0 for sets with nothing in common.

**By hand**, with $k = 3$: *the fox* has the shingles {`the`, `he␣`, `e␣f`, `␣fo`, `fox`} and *the fix* has {`the`, `he␣`, `e␣f`, `␣fi`, `fix`}, where ␣ marks a space. Three shingles are shared and seven are distinct, so $J = 3/7 \approx 0.43$.

A small $k$ makes unrelated texts look similar, because short substrings such as `the` occur everywhere; a large $k$ lets one changed word destroy many shingles. Comparing all pairs costs $n(n-1)/2$ similarity computations, about 137,000 here. At web scale, MinHash and locality-sensitive hashing estimate the same similarity far more cheaply (Exercise 2).

In [ ]:
word = "banana"
windows = [word[i:i + 3] for i in range(len(word) - 3 + 1)]
print(windows, "->", set(windows))  # four windows, but a set keeps each shingle once

### ✍️ Your turn: `char_shingles`

Write `char_shingles(text, k)`, which returns the **set** of all substrings of length `k` of `normalise(text)`.

- If the normalised text is shorter than `k` but not empty, return a set containing just that text, so that very short documents can still be compared.
- If the normalised text is empty, return an empty set.

Hint: a text of length $L \ge k$ has $L - k + 1$ windows, starting at `i = 0, 1, …, L - k`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def char_shingles_reference(text, k):
    text = normalise(text)
    if not text:
        return set()
    if len(text) < k:
        return {text}
    return {text[i:i + k] for i in range(len(text) - k + 1)}

In [ ]:
def char_shingles(text, k):
    """Set of every k-character substring of normalise(text); {text} if it is shorter than k; set() if empty."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
char_shingles = practice("char_shingles", char_shingles, char_shingles_reference,
                         cases=[("The fox", 3), ("aaaa", 2), ("ab", 5), ("   ", 3)])

### ✍️ Your turn: `jaccard`

Write `jaccard(a, b)` for two sets: the size of their intersection divided by the size of their union, as a float. Two empty sets are identical, so return `1.0` for them.

Hint: Python sets support `a & b` (intersection) and `a | b` (union).

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def jaccard_reference(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

In [ ]:
def jaccard(a, b):
    """|a & b| / |a | b| as a float; 1.0 when both sets are empty."""
    # ✍️ Your code here (about three lines)
    raise NotImplementedError

In [ ]:
jaccard = practice("jaccard", jaccard, jaccard_reference,
                   cases=[({1, 2, 3}, {2, 3, 4}), ({"a"}, set()), (set(), set()), ({"x", "y"}, {"y", "x"})])
print("the fox / the fix:", jaccard(char_shingles("the fox", 3), char_shingles("the fix", 3)), "= 3/7 =", 3 / 7)

**Expected behaviour.** `{1, 2, 3}` and `{2, 3, 4}` share two of their four distinct elements, so their similarity is 0.5; any set compared with the empty set scores 0; and the worked example prints $3/7 \approx 0.4286$.

Now compare every pair of the remaining documents with 5-character shingles. The matrix of similarities is computed once and reused by the experiment below. Documents are then visited in id order, and a document is dropped when it is at least `THRESHOLD` similar to one already kept, so the original (with the smaller id) survives and `near_replaced_by` records the decision.

In [ ]:
SHINGLE_SIZE, THRESHOLD = 5, 0.7


def similarity_matrix(documents, k):
    """Jaccard similarity of the k-shingle sets of every pair of documents."""
    sets = [char_shingles(d["text"], k) for d in documents]
    sim = np.eye(len(sets))
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            sim[i, j] = sim[j, i] = jaccard(sets[i], sets[j])
    return sim


def drop_near_duplicates(documents, sim, threshold):
    """Keep documents in order; drop one that is at least `threshold` similar to an earlier kept one."""
    kept, replaced = [], {}
    for i, doc in enumerate(documents):
        closest = max(kept, key=lambda j: sim[i, j], default=None)
        if closest is not None and sim[i, closest] >= threshold:
            replaced[doc["doc_id"]] = documents[closest]["doc_id"]
        else:
            kept.append(i)
    return [documents[i] for i in kept], replaced


start = time.perf_counter()
sim5 = similarity_matrix(docs_unique, SHINGLE_SIZE)
docs_distinct, near_replaced_by = drop_near_duplicates(docs_unique, sim5, THRESHOLD)
print(f"{len(docs_unique) * (len(docs_unique) - 1) // 2:,} pairs compared in {time.perf_counter() - start:.2f} s")
print(f"{len(docs_unique)} -> {len(docs_distinct)} documents: {len(near_replaced_by)} near-duplicates removed")
near_recall = float(np.mean([c in near_replaced_by for c in injected["near"]]))
print(f"injected near-copies caught: {near_recall:.0%}")
position = {d["doc_id"]: i for i, d in enumerate(docs_unique)}
for removed, kept in list(near_replaced_by.items())[:3]:
    print(f"\nJ = {sim5[position[removed], position[kept]]:.2f}")
    print(f"  kept    {kept}: {docs_unique[position[kept]]['text'][:80]}")
    print(f"  removed {removed}: {docs_unique[position[removed]]['text'][:80]}")

### 🧪 Experiment: choose the threshold from measurements

Which threshold is right? Because we know which documents are injected near-copies, we can count, for each threshold, how many of them are caught and how many *other* documents are dropped. Run the cell, then set `shingle_size` to 3 and to 8 and run it again. Any size other than 5 recomputes the similarity matrix, which takes about half a second on a laptop.

In [ ]:
shingle_size = 5  # 🧪 try 3 and 8, then run this cell again
sim = sim5 if shingle_size == SHINGLE_SIZE else similarity_matrix(docs_unique, shingle_size)
print(f"shingle size {shingle_size}")
print("threshold  dropped  injected near-copies caught  other documents dropped")
for threshold in (0.5, 0.6, 0.7, 0.8, 0.9):
    _, replaced = drop_near_duplicates(docs_unique, sim, threshold)
    caught = sum(c in replaced for c in injected["near"])
    print(f"{threshold:>9}  {len(replaced):>7}  {caught:>17} of {len(injected['near'])}  {len(replaced) - caught:>23}")

**What to notice.** Low thresholds catch every injected near-copy but also drop documents that are merely alike: with 5-character shingles at 0.5, about two dozen distinct stories and shopping lists go. High thresholds keep those but miss edited copies, because changing one word of a short document destroys several shingles at once. Neither error is free: false alarms throw away real data, and misses leave repeated text in. Look at a few of the "other" pairs before deciding; some are genuine near-duplicates that the generator produced by chance, which no list of injected documents contains. With 3-character shingles almost everything looks alike, because short substrings such as `the` are everywhere.

We keep $k = 5$ and a threshold of 0.7 for the rest of the notebook.

## 4 · Evaluation contamination: look for evaluation text in the training data

If evaluation sentences sit in the training data, a model can score well by remembering rather than by generalising, and the evaluation no longer measures what it claims to. A standard screen is **n-gram overlap**: collect every run of $n$ consecutive words in the evaluation items, and flag any training document $d$ that contains one of them,

$$\text{flag}(d) \iff \text{ngrams}_n(d) \cap \bigcup_{e \in E} \text{ngrams}_n(e) \neq \varnothing.$$

The choice of $n$ decides what you see. The GPT-3 report, for example, screened its benchmarks with 13-word overlaps. Our evaluation items, such as *kqo backwards is oqk.*, have only four words, so $n$ must be at most 4, and a small $n$ matches common phrases such as *backwards is* that every letter game shares. The check is also blind to paraphrases: the same fact in other words shares no long run of words.

When a document is flagged, remove it from the training data. Never edit the evaluation set to make the numbers look clean.

In [ ]:
print(re.findall(r"[a-z0-9]+", "Say ABC twice: abc abc.".lower()))  # words are runs of letters or digits

### ✍️ Your turn: `word_ngrams`

Write `word_ngrams(text, n)`, which returns the **set** of tuples of `n` consecutive words of `text`.

- The words are `re.findall(r"[a-z0-9]+", text.lower())`, as in the demo.
- A text with fewer than `n` words has no n-grams, so return an empty set.

Hint: build `tuple(words[i:i + n])` for `i` in `range(len(words) - n + 1)`; when that range is empty, a set comprehension over it is empty too.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def word_ngrams_reference(text, n):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}

In [ ]:
def word_ngrams(text, n):
    """Set of tuples of n consecutive words (runs of a-z or 0-9 in the lower-cased text)."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
word_ngrams = practice("word_ngrams", word_ngrams, word_ngrams_reference,
                       cases=[("The cat sat.", 2), ("one two", 3), ("a b a b", 2), ("kqo backwards is oqk.", 4)])

**Expected behaviour.** *The cat sat.* gives `{("the", "cat"), ("cat", "sat")}`; two words have no 3-grams; repeated pairs appear once in the set; and a four-word evaluation item has exactly one 4-gram, itself.

Now screen the remaining documents against the 40 evaluation sentences with $n = 3$.

In [ ]:
EVAL_ITEMS = [f"{s} backwards is {s[::-1]}." for s in EVAL_STRINGS]
N_GRAM = 3


def contaminated(documents, items, n):
    """Ids of the documents that share at least one word n-gram with any evaluation item."""
    eval_ngrams = set().union(*(word_ngrams(item, n) for item in items))
    return [d["doc_id"] for d in documents if word_ngrams(d["text"], n) & eval_ngrams]


flagged = contaminated(docs_distinct, EVAL_ITEMS, N_GRAM)
print("flagged:", flagged)
print("injected leaks caught:", sum(q in flagged for q in injected["leaks"]), "of", len(injected["leaks"]))
paraphrase = next(d for d in docs_distinct if d["doc_id"] == injected["paraphrase"])
print(f"paraphrased leak flagged: {paraphrase['doc_id'] in flagged}   ({paraphrase['doc_id']}: {paraphrase['text']})")

### 🧪 Experiment: how the n-gram size changes what you catch

Screen the same documents with $n$ from 2 to 5 and compare.

In [ ]:
print("n  flagged  injected leaks caught  paraphrase caught")
for n in (2, 3, 4, 5):
    hits = contaminated(docs_distinct, EVAL_ITEMS, n)
    caught = sum(q in hits for q in injected["leaks"])
    print(f"{n}  {len(hits):>7}  {caught:>16} of {len(injected['leaks'])}  {str(injected['paraphrase'] in hits):>17}")

**What to notice.** With $n = 2$ the screen flags about a hundred documents, almost all of them false alarms that merely contain *backwards is*; it does catch the paraphrase, but only by luck, through the pair *is* followed by the reversed string. With $n = 3$ or $4$ it flags exactly the four quoted leaks. With $n = 5$ it flags nothing, because no evaluation item is five words long: a silent failure worth remembering. No setting catches the paraphrase without drowning it in false alarms.

A second, targeted check helps. We know the reserved strings themselves, and they are rare, so we can search the training documents for any of them, or their reversals (the evaluation answers), as a whole word. The generator already avoids both, so only leaked documents should turn up. Layering a cheap, broad screen with a targeted search for the evaluation answers is common practice; neither check alone certifies a clean corpus.

In [ ]:
mentions = [d["doc_id"] for d in docs_distinct if RESERVED & set(re.findall(r"[a-z]+", d["text"]))]
print("documents that mention a reserved string or its reversal:", mentions)

removed = set(flagged) | set(mentions)
corpus = [d for d in docs_distinct if d["doc_id"] not in removed]
print(f"{len(docs_distinct)} -> {len(corpus)} documents after removing contaminated ones")
print(f"overall: {len(docs)} raw documents -> {len(corpus)} clean documents")

## 5 · Tokenise, split by document, then pack

The model reads **tokens**. Our tokeniser is as simple as possible: one token per character, plus five special tokens. `<pad>` fills unused positions, `<eos>` ends a document, and `<|user|>`, `<|assistant|>` and `<|end|>` are reserved now for the chat template of notebook 02, just as real tokenisers reserve their special tokens before pretraining.

Two decisions come before any window of tokens is cut.

- **Split by document first.** If you cut the stream into windows and then split the *windows* at random, neighbouring windows from one document land on both sides, and validation measures memory instead of generalisation. Assign whole documents to training or validation, then cut windows inside each split.
- **Pack.** Instead of padding every document to a fixed length, put the documents end to end, each followed by `<eos>`, and cut windows of `CONTEXT + 1` tokens: the first `CONTEXT` are the inputs and the last `CONTEXT` the next-token targets. Packing wastes no positions on padding, but attention can now look across a document boundary into the previous document. That is a decision to record; Exercise 1 builds the mask that forbids it.

In [ ]:
# ── Vocabulary and tokeniser (notebook 01) ─────────────────────────────────
# One token per character, plus five special tokens. The chat template's
# tokens are reserved from the start, as real tokenisers reserve theirs.
SPECIALS = ["<pad>", "<eos>", "<|user|>", "<|assistant|>", "<|end|>"]
VOCAB = SPECIALS + list(" .,:?0123456789abcdefghijklmnopqrstuvwxyz")
TOKEN_ID = {token: index for index, token in enumerate(VOCAB)}
PAD, EOS, USER, ASSISTANT, END = range(len(SPECIALS))


def encode(text):
    """Text -> list of token ids. Each special token, such as <|end|>, becomes a single id."""
    ids, i = [], 0
    while i < len(text):
        special = next((s for s in SPECIALS if text.startswith(s, i)), None)
        if special is not None:
            ids.append(TOKEN_ID[special])
            i += len(special)
        else:
            ids.append(TOKEN_ID[text[i]])
            i += 1
    return ids


def decode(ids):
    """List of token ids -> text."""
    return "".join(VOCAB[int(i)] for i in ids)

In [ ]:
def pack_documents(documents):
    """One token stream: every document followed by <eos>."""
    stream = []
    for doc in documents:
        stream += encode(doc["text"]) + [EOS]
    return np.array(stream)


order = np.random.default_rng(3).permutation(len(corpus))
n_val = len(corpus) // 8
val_docs = [corpus[i] for i in order[:n_val]]
train_docs = [corpus[i] for i in order[n_val:]]
assert not {d["doc_id"] for d in train_docs} & {d["doc_id"] for d in val_docs}

train_stream, val_stream = pack_documents(train_docs), pack_documents(val_docs)
print(f"training:   {len(train_docs)} documents -> {len(train_stream):,} tokens")
print(f"validation: {len(val_docs)} documents -> {len(val_stream):,} tokens")
print(f"vocabulary: {len(VOCAB)} tokens")
print("one packed window:", repr(decode(train_stream[:CONTEXT + 1])))
print()
print("source     documents   tokens   share of training tokens")
for source in ("stories", "letters", "lists"):
    count = sum(d["source"] == source for d in train_docs)
    tokens = sum(len(encode(d["text"])) + 1 for d in train_docs if d["source"] == source)
    print(f"{source:<9} {count:>9} {tokens:>8} {tokens / len(train_stream):>17.0%}")

**What to notice.** Count tokens as well as documents. The stories are a little under half of the documents but more than half of the tokens, because they are longer; the shopping lists are short, so they contribute even less of the training signal than their document count suggests. A mixture is set in tokens, and a large source can dominate it even when document counts look balanced.

## 6 · Plan the compute before you train

A training run is planned in **tokens** and **floating-point operations** (FLOPs), not in epochs. For a dense transformer with $N$ parameters, the forward pass costs about $2N$ FLOPs per token, because each weight takes part in one multiplication and one addition, and the backward pass costs about twice as much again. Training on $D$ tokens therefore costs approximately

$$C \approx 6ND \ \text{FLOPs}.$$

This is the estimate used by Kaplan et al. (2020), *Scaling Laws for Neural Language Models*. It assumes a dense model, counts $N$ without the embedding tables, and leaves out the attention-score computation, which adds about $6 \cdot n_\text{layer} \cdot n_\text{ctx} \cdot d_\text{model}$ FLOPs per token: well under 1% for a large model with a short context, but about 8% for our tiny one.

Two more quantities complete a plan:

- **Tokens per parameter**, $D / N$. Hoffmann et al. (2022), *Training Compute-Optimal Large Language Models* (the Chinchilla paper), found that for a fixed compute budget their models did best with roughly 20 training tokens per parameter. That ratio comes from their models and data, and assumes the tokens are fresh rather than repeated.
- **Wall-clock time**: $C$ divided by the hardware's peak rate times the fraction of it you actually achieve, the *utilisation*.

Measure a short pilot run before trusting any of these numbers; the demo also shows how the tokens in one optimiser update multiply out.

In [ ]:
workers, sequences_per_microbatch, accumulation_steps, tokens_per_sequence = 4, 2, 8, 1024
print("tokens per update:", workers * sequences_per_microbatch * accumulation_steps * tokens_per_sequence)
print(f"a 1-billion-parameter model on 20 billion tokens: 6ND = {6 * 1e9 * 20e9:.1e} FLOPs")

### ✍️ Your turn: `training_budget`

Write `training_budget(n_params, n_tokens, peak_flops_per_second, utilisation)`, which returns a dict with three keys:

- `"flops"`: $6 \times$ `n_params` $\times$ `n_tokens`;
- `"tokens_per_parameter"`: `n_tokens / n_params`;
- `"hours"`: the FLOPs divided by the achieved rate, `peak_flops_per_second * utilisation`, converted from seconds to hours.

Hint: compute the FLOPs once and reuse them.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def training_budget_reference(n_params, n_tokens, peak_flops_per_second, utilisation):
    flops = 6 * n_params * n_tokens
    return {"flops": flops,
            "tokens_per_parameter": n_tokens / n_params,
            "hours": flops / (peak_flops_per_second * utilisation) / 3600}

In [ ]:
def training_budget(n_params, n_tokens, peak_flops_per_second, utilisation):
    """{"flops": 6 N D, "tokens_per_parameter": D / N, "hours": flops / (peak * utilisation) / 3600}."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
training_budget = practice("training_budget", training_budget, training_budget_reference,
                           cases=[(1e9, 20e9, 1e15, 0.4), (70e9, 1.4e12, 1e15, 0.4), (26_688, 153_600, 1e10, 0.5)])

**Expected behaviour.** The 1-billion-parameter model on 20 billion tokens needs $1.2 \times 10^{20}$ FLOPs, which at a hypothetical peak of $10^{15}$ FLOP/s and 40% utilisation is about 83 hours on one accelerator. The Chinchilla model itself, 70 billion parameters trained on 1.4 trillion tokens, needs about $5.9 \times 10^{23}$ FLOPs, with exactly 20 tokens per parameter.

Now apply the calculator to our own run: 300 steps, each on 16 windows of 32 positions.

In [ ]:
STEPS, BATCH = 300, 16
probe = init_tiny_gpt(seed=0)
n_total = sum(t.data.size for t in probe.values())
n_params = n_total - probe["tok"].data.size - probe["pos"].data.size  # without the embedding tables
n_tokens = STEPS * BATCH * CONTEXT
plan = training_budget(n_params, n_tokens, peak_flops_per_second=1e10, utilisation=1.0)
print(f"parameters: {n_total:,} in total, {n_params:,} without the embedding tables")
print(f"tokens: {STEPS} steps x {BATCH} windows x {CONTEXT} positions = {n_tokens:,}")
print(f"6ND = {plan['flops']:.2e} FLOPs; {plan['tokens_per_parameter']:.1f} tokens per parameter")
print(f"our run reads the training stream about {n_tokens / len(train_stream):.1f} times")
print(f"20 tokens per parameter would be {20 * n_params:,} tokens: "
      f"{20 * n_params / len(train_stream):.0f} passes over our {len(train_stream):,} training tokens")

**What to notice.** Our run plans about 6 tokens per parameter, well short of the Chinchilla ratio, and even so it reads the training stream about five times. Reaching 20 tokens per parameter would mean about 17 passes over the same small corpus, and repeated tokens are worth less than fresh ones. The heuristic is a statement about fresh data at large scale; with a small fixed corpus, the data, not the compute, limits what the model can learn. The aim of this run is a quick, measurable demonstration, not a compute-optimal model.

## 7 · Pretrain the tiny model

Each step draws 16 random windows of 33 tokens from the training stream, predicts tokens 2 to 33 from tokens 1 to 32, and updates every weight with Adam. Every 50 steps we compute the loss on all the validation windows, which never change, and keep a copy of the weights with the best validation loss: a checkpoint you could reload.

To judge the numbers, compare them with simple baselines estimated from the training stream: a **uniform** guess over the vocabulary ($\ln |V|$ nats per token), a **unigram** model that predicts each token from its overall frequency, and a **bigram** model that predicts from the previous token only. Our transformer sees up to 32 previous tokens, so it should beat the bigram model clearly.

In [ ]:
def windows_of(stream):
    """Non-overlapping windows of CONTEXT + 1 tokens covering the stream."""
    count = (len(stream) - 1) // CONTEXT
    return np.stack([stream[i * CONTEXT: i * CONTEXT + CONTEXT + 1] for i in range(count)])


val_windows = windows_of(val_stream)
V = len(VOCAB)
unigram = (np.bincount(train_stream, minlength=V) + 1.0) / (len(train_stream) + V)  # add-one smoothing
bigram = np.ones((V, V))  # add-one smoothing as well
np.add.at(bigram, (train_stream[:-1], train_stream[1:]), 1.0)
bigram /= bigram.sum(axis=1, keepdims=True)
previous, targets = val_windows[:, :-1], val_windows[:, 1:]
baselines = {"uniform": float(np.log(V)), "unigram": float(-np.log(unigram[targets]).mean()),
             "bigram": float(-np.log(bigram[previous, targets]).mean())}
print("validation loss of the baselines (nats per token):", {k: round(v, 3) for k, v in baselines.items()})

In [ ]:
learning_rate = 3e-3  # 🧪 try 3e-2 and 3e-4, then run this cell and the next three again

In [ ]:
params = init_tiny_gpt(seed=0)
optimiser = Adam(list(params.values()), lr=learning_rate)
window_picker = np.random.default_rng(1)
history = {"step": [], "train": [], "val": []}
best = {"val": np.inf, "step": 0, "weights": None}
start = time.perf_counter()
for step in range(1, STEPS + 1):
    starts = window_picker.integers(0, len(train_stream) - CONTEXT - 1, BATCH)
    batch = np.stack([train_stream[s:s + CONTEXT + 1] for s in starts])
    loss = cross_entropy(gpt_forward(params, batch[:, :-1]), batch[:, 1:])
    zero_grad(optimiser.tensors)
    loss.backward()
    optimiser.step()
    if step == 1 or step % 50 == 0:
        val_loss = float(cross_entropy(gpt_forward(params, val_windows[:, :-1]), val_windows[:, 1:]).data)
        history["step"].append(step)
        history["train"].append(float(loss.data))
        history["val"].append(val_loss)
        if val_loss < best["val"]:
            best = {"val": val_loss, "step": step, "weights": {k: t.data.copy() for k, t in params.items()}}
        print(f"step {step:>3}: training batch loss {float(loss.data):.3f}, validation loss {val_loss:.3f}")
seconds = time.perf_counter() - start
print(f"{STEPS} steps in {seconds:.1f} s; best validation loss {best['val']:.3f} at step {best['step']}")

In [ ]:
plt.figure(figsize=(6.5, 3.6))
plt.plot(history["step"], history["train"], "o-", label="training batch")
plt.plot(history["step"], history["val"], "o-", label="validation (held-out documents)")
for name, value in baselines.items():
    plt.axhline(value, linestyle=":", color="grey")
    plt.text(STEPS, value, f" {name}", va="center", fontsize=8)
plt.xlabel("step")
plt.ylabel("loss (nats per token)")
plt.legend()
plt.tight_layout()
plt.show()

A single average can hide a problem in one source, so restore the best checkpoint and report the validation loss of each source separately. Then sample some text: start from *the␣* and repeatedly draw the next character from the model's predicted distribution. Dividing the logits by a **temperature** below 1 before the softmax sharpens that distribution towards the likeliest characters.

In [ ]:
for name, tensor in params.items():
    tensor.data[...] = best["weights"][name]  # restore the best checkpoint
print("validation loss by source (nats per token):")
for source in ("stories", "letters", "lists"):
    w = windows_of(pack_documents([d for d in val_docs if d["source"] == source]))
    print(f"  {source:<8} {float(cross_entropy(gpt_forward(params, w[:, :-1]), w[:, 1:]).data):.3f}   ({len(w)} windows)")

temperature = 0.7  # below 1 sharpens the distribution; try 1.0
sampler = np.random.default_rng(4)
tokens = encode("the ")
for _ in range(90):
    logits = gpt_forward(params, np.array([tokens[-CONTEXT:]])).data[0, -1] / temperature
    probs = np.exp(logits - logits.max())
    tokens.append(int(sampler.choice(V, p=probs / probs.sum())))
print("\nsampled text:", repr(decode(tokens)))

**What to notice.**

- The validation loss ends far below the bigram baseline, so the model uses more than the previous character. It started near the uniform baseline, because an untrained model spreads its probability over the whole vocabulary.
- By source, the stories are the easiest: their grammar is rigid and their vocabulary small. The letter games are the hardest, because a random string costs about $\ln 26 \approx 3.3$ nats for each of its new letters, a floor that no model can beat; only the repeats in *say yad twice: yad yad* are predictable. A falling average can hide a source whose loss stalls, which is why you report each one.
- The sample has the frame of a story sentence (*the … the river.*) and many correctly spelled toy-world words, but also misspellings and no meaning. That is what 300 steps of a 29,000-parameter model can learn from 31,000 tokens. Real base models differ in scale (billions of parameters, trillions of tokens), data (curated text from many sources and languages) and hardware (clusters of accelerators running for weeks), not in the loop you have just run.

Finally, compare the plan with the measurement. The achieved rate includes the validation passes and the Python overhead of the tiny tensor engine.

In [ ]:
print(f"planned 6ND = {plan['flops']:.2e} FLOPs in {seconds:.1f} s: about {plan['flops'] / seconds / 1e9:.1f} GFLOP/s achieved")

## Put it together

The cell below re-checks every promise this notebook made about the data and the model. If a change above breaks one of them, for example a learning rate from the 🧪 cell that makes training diverge, it stops here with a message.

In [ ]:
# Exact duplicates: one document per normalised text, and every injected copy removed.
digests = [content_hash(d["text"]) for d in corpus]
assert len(set(digests)) == len(digests), "two documents still share a normalised text"
assert all(c in replaced_by for c in injected["exact"]), "an injected exact copy survived"

# Near-duplicates: no remaining pair at or above the threshold, and most injected near-copies caught.
kept_positions = [position[d["doc_id"]] for d in corpus]
assert (sim5[np.ix_(kept_positions, kept_positions)] - np.eye(len(corpus))).max() < THRESHOLD
assert near_recall >= 0.6, f"only {near_recall:.0%} of the injected near-copies were caught"

# Contamination: no evaluation 3-gram and no reserved string left in the training data.
assert not contaminated(corpus, EVAL_ITEMS, N_GRAM)
assert not any(RESERVED & set(re.findall(r"[a-z]+", d["text"])) for d in corpus)

# Split: whole documents on one side only.
assert not {d["doc_id"] for d in train_docs} & {d["doc_id"] for d in val_docs}

# Training: validation loss fell by at least 1 nat and beats the bigram baseline by at least 0.5 nats.
assert history["val"][-1] < history["val"][0] - 1.0, "validation loss did not fall; did you change learning_rate?"
assert best["val"] < baselines["bigram"] - 0.5, "the transformer does not beat the bigram baseline by 0.5 nats"

# The plan: 6ND, by definition.
assert np.isclose(plan["flops"], 6 * n_params * n_tokens)

print("✅ All checks passed.")
print(f"{len(docs)} raw documents -> {len(corpus)} clean documents; best validation loss {best['val']:.3f} nats per token "
      f"(bigram {baselines['bigram']:.3f}, uniform {baselines['uniform']:.3f})")

## Exercises

These go beyond the type-alongs. The first two have checks; the last two are questions to answer in a markdown cell of your own.

### Exercise 1 ✍️ `document_attention_mask`: keep attention inside each document

Packing lets attention look across `<eos>` into the previous document. Write `document_attention_mask(ids, eos_id)` for one packed window `ids` of length $T$. It returns an additive mask of shape `(T, T)`, like the causal mask inside `gpt_forward`: `0.0` where query position $i$ may attend to key position $j$, and `-1e9` elsewhere. Position $i$ may attend to $j$ when $j \le i$ **and** both positions belong to the same document. An `<eos>` token belongs to the document it ends, so the token after it starts a new document.

Hint: the document number of position $t$ is the number of `<eos>` tokens *before* $t$: `np.concatenate([[0], np.cumsum(ids[:-1] == eos_id)])`.

In [ ]:
def document_attention_mask_reference(ids, eos_id):
    ids = np.asarray(ids)
    document = np.concatenate([[0], np.cumsum(ids[:-1] == eos_id)])
    allowed = (document[:, None] == document[None, :]) & np.tril(np.ones((len(ids), len(ids)), dtype=bool))
    return np.where(allowed, 0.0, -1e9)

In [ ]:
def document_attention_mask(ids, eos_id):
    """(T, T) additive mask: 0.0 where j <= i and j is in the same document as i, else -1e9."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
document_attention_mask = practice("document_attention_mask", document_attention_mask, document_attention_mask_reference,
                                   cases=[([5, 6, EOS, 7, 8], EOS), ([5, 6, 7], EOS), ([EOS, EOS, 9], EOS)])
print((document_attention_mask([5, 6, EOS, 7, 8], EOS) == 0).astype(int))  # 1 = may attend

To use the mask in training you would pass it to the attention in place of the plain causal mask, one mask per window. Would you expect the validation loss to change much on this corpus? Think about how much of a document's next character depends on the *previous* document.

### Exercise 2 ✍️ `minhash_signature`: estimate Jaccard similarity cheaply

Comparing all pairs grows with the square of the corpus size. **MinHash** summarises each shingle set by a short signature: for each of `n_hashes` different hash functions, keep the smallest hash value over the set. For one random hash function, the chance that two sets have the same minimum equals their Jaccard similarity, so the fraction of matching signature entries estimates $J$. Locality-sensitive hashing then groups documents by parts of their signatures, so that only likely pairs are compared at all.

Write `minhash_signature(shingles, n_hashes)`, which returns a list of `n_hashes` integers. Entry `i` is the minimum over the shingles `s` of `int(hashlib.sha256(f"{i}:{s}".encode("utf-8")).hexdigest()[:8], 16)`. For an empty set, return `[-1] * n_hashes`.

In [ ]:
def minhash_signature_reference(shingles, n_hashes):
    if not shingles:
        return [-1] * n_hashes
    return [min(int(hashlib.sha256(f"{i}:{s}".encode("utf-8")).hexdigest()[:8], 16) for s in shingles)
            for i in range(n_hashes)]

In [ ]:
def minhash_signature(shingles, n_hashes):
    """For i in range(n_hashes): min over s of int(sha256(f"{i}:{s}").hexdigest()[:8], 16); [-1] * n_hashes if empty."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
minhash_signature = practice("minhash_signature", minhash_signature, minhash_signature_reference,
                             cases=[({"abc", "bcd"}, 4), (set(), 3), ({"the"}, 2)])

removed_id, kept_id = next(iter(near_replaced_by.items()))
a = char_shingles(docs_unique[position[kept_id]]["text"], 5)
b = char_shingles(docs_unique[position[removed_id]]["text"], 5)
sig_a, sig_b = minhash_signature(a, 128), minhash_signature(b, 128)
print(f"{kept_id} vs {removed_id}: exact J = {jaccard(a, b):.3f}, "
      f"MinHash estimate from 128 hashes = {np.mean(np.array(sig_a) == np.array(sig_b)):.3f}")

### Exercise 3 (question): set the mixture on purpose

Suppose the shopping lists mattered most to the application this model will serve. Using the token shares from section 5, propose new mixture weights, explain how you would apply them (repeating some documents, sampling sources with fixed probabilities, or collecting more), and name two measurements you would watch to check that the change helped without harming the other sources.

### Exercise 4 (question): what does a checkpoint need?

The `best` dictionary stores only weights. List everything else you would have to save to *resume* this run exactly at step 150 rather than merely reuse the weights: think about the state inside `Adam`, the window sampler, the step counter and the data. Then explain why a long run should test resuming from a checkpoint *before* it is needed.

## Common errors and debugging

- **Hashing raw text.** Different case or spacing produces different digests, so copies slip through. Normalise first, and use the same `normalise` everywhere.
- **"Keep the first" after shuffling.** The surviving copy is then arbitrary, often the mangled one. Use a fixed rule and record `replaced_by`.
- **Jaccard on lists instead of sets.** Repeated shingles inflate the counts and the result can exceed 1. Build sets.
- **An n-gram size larger than the evaluation items.** Nothing can match, and the screen silently reports a clean corpus. Check the length of your items and test the screen on a known leak, as you did here.
- **Splitting windows instead of documents.** Neighbouring windows of one document on both sides of the split make validation look better than it is.
- **Counting documents instead of tokens.** Mixture weights, budgets and tokens per parameter are all about tokens.
- **Comparing parameter counts on different conventions.** Published ratios such as 20 tokens per parameter use a particular count; say whether yours includes the embedding tables.
- **Keeping a reference instead of a copy.** `best["weights"] = params` would keep pointing at the weights as they keep changing; copy the arrays, as the training loop does.

## Summary

- **Exact duplicates:** normalise, hash with SHA-256, group by digest, keep a fixed representative and record what replaced what.
- **Near-duplicates:** compare character shingles with Jaccard similarity; choose $k$ and the threshold from measured catches and false alarms, not by habit.
- **Contamination:** screen training data against the evaluation set with word n-grams, choose $n$ to suit the length of the items, and add a targeted search because n-grams miss paraphrases. Remove training documents, never evaluation items.
- **Planning:** $C \approx 6ND$ FLOPs for a dense model, about 20 fresh tokens per parameter for compute-optimal training in the Chinchilla study, and a measured pilot before any long run.
- **Pretraining:** split by document, pack with `<eos>`, decide whether attention may cross documents, track validation loss per source against simple baselines, and keep a checkpoint.

## Next

In **notebook 02** you turn a pretrained base model into one that follows instructions. You write the chat template that uses the `<|user|>`, `<|assistant|>` and `<|end|>` tokens reserved here, decide which tokens the loss should score, and measure what happens when the prompt is scored too. Notebook 02 re-creates a base model with the recipe from this notebook, so you can start it directly.